# 05 — Entraînement — le pipeline réel, ses artefacts et sa reproductibilité

**Projet** : Résumé de comptes-rendus d'intervention — encodeur-décodeur appris, baseline extractive publiée

## Objectifs pédagogiques

1. Exécuter les pipelines du projet (`generate-data`, `train`) dans un bac à sable qui n'écrase aucun artefact de production.
1. Lire la mécanique d'un encodeur-décodeur : vocabulaire appris, pré-entraînement par débruitage, affinage supervisé, et ce que chaque étape change dans la perte.
1. Recharger l'artefact et vérifier qu'il résume **exactement** comme le modèle ajusté.
1. Prouver le déterminisme : deux ajustements à graine fixée donnent les mêmes résumés, jeton pour jeton.

## 0. Environnement et corpus

Les pipelines du projet tournent ici sur le corpus réduit du notebook et écrivent dans `outputs/notebooks/` : `make train` reste intact.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (200 comptes-rendus), un budget
# d'époques réduit et une architecture réduite : l'exécution reste de quelques secondes, et les
# valeurs absolues d'un notebook ne sont **pas** celles de la référence publiée par le README.
NB_DOCUMENTS = 200
NB_EPOCHS = 8

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "model.params.layers=1",
                "model.params.units=96",
                "model.params.pretraining.epochs=1",
                "+model.params.vocab_size=800",
                "+model.params.min_frequency=2",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
                "train.early_stopping.enabled=false",
                "seed=42",
                "log_level=WARNING",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Bac à sable : les notebooks génèrent **leur** corpus (réduit) et écrivent **leurs** artefacts sous
# `outputs/notebooks/`. `data/raw`, `artifacts/` et donc les chiffres publiés par le README restent
# ceux de `make data` / `make train` : aucun notebook ne peut les modifier par accident.
NB_ROOT = PROJECT_ROOT / "outputs" / "notebooks"
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=NB_ROOT / "data",
    artifacts_dir=NB_ROOT / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
ID_COLUMN = str(MODEL_NODE.get("id_column", "doc_id"))
COMPRESSION = float(MODEL_NODE.get("params", {}).get("compression", 0.45))
MAX_OUTPUT_TOKENS = int(MODEL_NODE.get("params", {}).get("max_output_tokens", 90))
STRATEGIES = list(MODEL_NODE.get("strategies_to_compare", []) or ["lead"])

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.45)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Stratégies comparées : {', '.join(STRATEGIES)}")
print(f"Budget de longueur: compression {COMPRESSION:.2f}, plafond {MAX_OUTPUT_TOKENS} tokens")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticSummaryCorpusGenerator
from src.data.loaders import SummaryCorpusLoader

LOADER = SummaryCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS, REFERENCES, FACTS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(
        f"Corpus du notebook lu : {len(DOCUMENTS)} comptes-rendus, "
        f"{len(REFERENCES)} résumés de référence, {len(FACTS)} faits annotés"
    )
else:
    # Un clone frais n'a pas encore de corpus réduit : le notebook le génère, de façon déterministe,
    # dans son bac à sable. `make data` produit le corpus de référence (600 comptes-rendus) dans
    # `data/raw` — les deux ne se mélangent jamais.
    bundle = SyntheticSummaryCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, REFERENCES, FACTS = bundle.documents, bundle.queries, bundle.facts
    METADATA = bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_references(REFERENCES)
    LOADER.save_facts(FACTS)
    LOADER.save_metadata(METADATA)
    print(f"Corpus réduit généré dans outputs/notebooks/data : {len(DOCUMENTS)} comptes-rendus")

TRAIN_DOCUMENTS, TRAIN_REFERENCES, TRAIN_FACTS = LOADER.load_split("train")
VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS = LOADER.load_split("val")
TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS = LOADER.load_split("test")
SPLIT_SIZES = LOADER.split_sizes()
# Le test ne sert **jamais** à régler quoi que ce soit : les réglages se lisent sur la validation,
# le verdict sur le test, une seule fois.
GOLDEN = dict(zip(REFERENCES["doc_id"], REFERENCES["summary"], strict=False))
print("splits :", SPLIT_SIZES)
print("compression moyenne demandée :", round(float(REFERENCES["compression"].mean()), 4))

In [ ]:
from src.pipelines import DataGenerationPipeline, TrainPipeline

DATA_RESULT = DataGenerationPipeline(CONFIG, paths=NB_PATHS).run()
print(
    "generate-data :",
    "succès" if DATA_RESULT.succeeded else "échec",
    f"en {DATA_RESULT.duration_seconds:.2f}s",
)
for message in DATA_RESULT.messages:
    print("  -", message)

TRAIN_RESULT = TrainPipeline(CONFIG, paths=NB_PATHS).run()
print()
print(
    "train :",
    "succès" if TRAIN_RESULT.succeeded else "échec",
    f"en {TRAIN_RESULT.duration_seconds:.2f}s",
)
for message in TRAIN_RESULT.messages:
    print("  -", message)
assert TRAIN_RESULT.succeeded, TRAIN_RESULT.messages
OUTCOME = TRAIN_RESULT.payload

**Ce qu'il faut retenir**

- Le notebook appelle **les pipelines du projet**, pas une réimplémentation : ce qu'on lit ici est exactement ce que fait `python -m src.main mode=train`.
- Le bac à sable (`outputs/notebooks/`, données **et** artefacts) est la seule différence : un notebook ne doit jamais écraser un artefact de production, sinon la reproductibilité du run publié devient une illusion.
- Le pipeline refuse de tourner sans son artefact d'entrée (`mode=evaluate` sans modèle entraîné) : un rapport plausible sur un modèle vide serait pire qu'une erreur.

## 1. La mécanique du modèle

Trois étapes, dans l'ordre : apprendre le vocabulaire sur le train (documents **et** résumés), pré-entraîner le décodeur à reconstruire un résumé à partir d'un document bruité, puis affiner en enseignant-forcé.

In [ ]:
fit_metrics = {
    key: round(value, 4)
    for key, value in OUTCOME.fit_metrics.items()
    if key.startswith(("pretrain", "loss", "epochs"))
}
display(pd.Series(fit_metrics, name="valeur").to_frame())

model = OUTCOME.model
card = model.model_card()
print("architecture :", card["architecture"])
print("vocabulaire  :", card["vocabulary"])
print("entraînement :", {k: v for k, v in card["training"].items() if k != "history"})
print("paramètres   :", card["n_parameters"])

In [ ]:
history = pd.DataFrame(card["training"]["history"] or [])
if not history.empty:
    display(history.round(4))
    history.plot(title="Perte par époque (pré-entraînement puis affinage)", figsize=(8, 4))
    plt.ylabel("perte")
    plt.show()
else:
    print("historique vide : le run a été trop court pour journaliser une époque")

## 2. Les artefacts du run

Quatre fichiers répondent à quatre questions : le modèle (comment prédire), les métriques (ce que ça vaut), la fiche modèle (comment c'est fait), la configuration résolue (avec quels réglages exactement).

In [ ]:
from src.utils.io import read_json

artefact = NB_PATHS.models_dir / "summarizer.pt"
for name in ("summarizer.pt", "model_card.json", "resolved_config.json"):
    path = NB_PATHS.models_dir / name
    print(
        f"{name:24s} {'écrit' if path.exists() else 'ABSENT':6s} "
        f"({path.stat().st_size if path.exists() else 0} octets)"
    )
for name in ("training_metrics.json",):
    path = NB_PATHS.metrics_dir / name
    print(
        f"{name:24s} {'écrit' if path.exists() else 'ABSENT':6s} "
        f"({path.stat().st_size if path.exists() else 0} octets)"
    )
assert artefact.exists() and (NB_PATHS.models_dir / "model_card.json").exists()

resolved = read_json(NB_PATHS.models_dir / "resolved_config.json")
print("\nalgorithme servi dans la configuration résolue :", resolved["model"]["algorithm"])
print(
    "architecture résolue :",
    {k: resolved["model"]["params"][k] for k in ("layers", "units", "heads")},
)

## 3. Recharger l'artefact

Un artefact qui ne prédit pas exactement comme le modèle ajusté est un artefact cassé — et cela ne se voit qu'en comparant les deux, ligne à ligne.

In [ ]:
from src.models import load_model

reloaded = load_model(artefact, config=CONFIG.model_dump())
comparison_documents = TEST_DOCUMENTS.head(5)
before = model.summarize_frame(comparison_documents)
after = reloaded.summarize_frame(comparison_documents)
identical = bool((before["prediction"] == after["prediction"]).all())
print("résumés identiques après rechargement :", identical)
display(pd.DataFrame({"doc_id": after["doc_id"], "résumé": after["prediction"].str.slice(0, 90)}))
assert identical, "l'artefact rechargé ne résume pas comme le modèle ajusté"

**Ce qu'il faut retenir**

- Le rechargement est testé sur des résumés **générés**, pas sur une matrice de poids : c'est la sortie qui compte, et un décodage légèrement différent la changerait sans que personne ne le voie.
- Le vocabulaire fait partie de l'artefact : un vocabulaire appris hors du modèle serait un second artefact à versionner, donc une source de désynchronisation.
- L'architecture est écrite dans la fiche modèle : un artefact se relit **et** s'explique sans le code qui l'a produit.

## 4. Déterminisme

Deux ajustements à graine fixée, sur le même split, doivent produire les mêmes résumés. Sans cette preuve, aucune comparaison entre deux runs n'est interprétable.

In [ ]:
from src.models import build_model

first = build_model(CONFIG, algorithm="transformer_tiny")
second = build_model(CONFIG, algorithm="transformer_tiny")
first.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES)
second.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES)

sample = TEST_DOCUMENTS.head(3)
first_summaries = first.summarize_frame(sample)["prediction"].tolist()
second_summaries = second.summarize_frame(sample)["prediction"].tolist()
for index, (left, right) in enumerate(zip(first_summaries, second_summaries, strict=True), start=1):
    print(f"{index}. {'identique' if left == right else 'DIFFÉRENT'} — {left[:80]}")
assert first_summaries == second_summaries, "deux ajustements à graine fixée divergent"
print("\ndéterminisme vérifié sur", len(sample), "résumés")

**Ce qu'il faut retenir**

- La graine est posée sur les trois générateurs (`random`, `numpy`, `torch`) **et** le nombre de fils d'exécution est fixé à un : les réductions flottantes changent l'ordre des additions, donc les derniers chiffres d'une perte.
- Le décodage est glouton : aucune température, aucun échantillonnage. Un résumé reproductible est la condition pour qu'un rapport d'évaluation soit vérifiable.
- La comparaison porte sur le texte produit, pas sur la perte : deux pertes égales à la troisième décimale peuvent donner deux résumés différents après un `argmax`.